# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

PRBS_START_SEC = 300
PRBS_END_SEC = 1000
TARGET_WR = 80.0

def find_column(df: pd.DataFrame, target: str) -> str:
    lowered = target.lower()
    for column in df.columns:
        if column.lower() == lowered:
            return column
    for column in df.columns:
        if column.lower().startswith(lowered):
            return column
    for column in df.columns:
        if lowered in column.lower():
            return column
    raise KeyError(f"Column '{target}' not found in dataframe.")

def ensure_one_hz(time: np.ndarray, signals: list[np.ndarray]) -> tuple[np.ndarray, list[np.ndarray]]:
    time = np.asarray(time, dtype=float)
    signals = [np.asarray(sig, dtype=float) for sig in signals]

    finite_mask = np.isfinite(time)
    for sig in signals:
        finite_mask &= np.isfinite(sig)
    time = time[finite_mask]
    signals = [sig[finite_mask] for sig in signals]
    if time.size < 2:
        raise ValueError("Insufficient samples to enforce 1 Hz grid.")

    order = np.argsort(time)
    time = time[order]
    signals = [sig[order] for sig in signals]

    time = time - np.nanmin(time)
    diffs = np.diff(time)
    valid_diffs = diffs[(diffs > 0) & np.isfinite(diffs)]
    dt = np.median(valid_diffs) if valid_diffs.size else 1.0
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Unable to determine sampling interval for 1 Hz resampling.")

    if abs(dt - 1.0) > 1e-2:
        end_time = time[-1]
        resample_times = np.arange(0.0, end_time + 1.0, 1.0)
        resampled = [np.interp(resample_times, time, sig) for sig in signals]
        return resample_times, resampled
    return time, signals

def load_subject_signals(data_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_csv(data_path)
    time_col = find_column(df, "Time")
    wr_col = find_column(df, "WorkRate")
    vo2_col = find_column(df, "VO2pulm")
    subset = df[[time_col, wr_col, vo2_col]].dropna()
    if subset.empty:
        raise ValueError("No valid samples after dropping NaNs.")
    time = subset[time_col].to_numpy(dtype=float)
    wr = subset[wr_col].to_numpy(dtype=float)
    vo2 = subset[vo2_col].to_numpy(dtype=float)
    time_aligned, [wr_aligned, vo2_aligned] = ensure_one_hz(time, [wr, vo2])
    return time_aligned, wr_aligned, vo2_aligned

def load_actual_vo2peak(target_path: Path) -> float:
    targets_df = pd.read_csv(target_path)
    column = find_column(targets_df, "VO2peak_abs")
    return float(targets_df[column].iloc[0])

def compute_vo2_80w(time: np.ndarray, work_rate: np.ndarray, vo2: np.ndarray) -> float:
    prbs_mask = (time >= PRBS_START_SEC) & (time <= PRBS_END_SEC)
    wr_mask = work_rate == TARGET_WR
    combined_mask = prbs_mask & wr_mask
    if combined_mask.sum() == 0:
        raise ValueError("No samples found for 80 W phase.")
    return float(np.nanmean(vo2[combined_mask]))

In [ ]:
data_files = sorted(PROCESSED_DIR.glob("*_data.csv"))
target_files = {path.name.replace("_targets.csv", ""): path for path in PROCESSED_DIR.glob("*_targets.csv")}

records = []
for data_path in data_files:
    subject_id = data_path.name.replace("_data.csv", "")
    target_path = target_files.get(subject_id)
    if target_path is None:
        print(f"{subject_id}: missing target file; skipping.")
        continue
    try:
        time, work_rate, vo2 = load_subject_signals(data_path)
        vo2_80w = compute_vo2_80w(time, work_rate, vo2)
        vo2peak = load_actual_vo2peak(target_path)
    except Exception as exc:
        print(f"{subject_id}: {exc}; skipping.")
        continue
    records.append({"subject": subject_id, "vo2_80w": vo2_80w, "vo2peak": vo2peak})

if not records:
    raise RuntimeError("No subjects had valid 80 W phase measurements.")
subject_df = pd.DataFrame(records)
subject_df

In [ ]:
predictions = []
for idx, row in subject_df.iterrows():
    subject_id = row['subject']
    holdout = subject_df[subject_df['subject'] == subject_id]
    training = subject_df[subject_df['subject'] != subject_id]
    if training.empty:
        print(f"{subject_id}: insufficient training data; skipping prediction.")
        continue
    mean_vo2_80w = training['vo2_80w'].mean()
    mean_vo2peak = training['vo2peak'].mean()
    if mean_vo2_80w <= 0:
        print(f"{subject_id}: invalid mean VO2_80W; skipping prediction.")
        continue
    scaling_factor = mean_vo2peak / mean_vo2_80w
    predicted_vo2peak = row['vo2_80w'] * scaling_factor
    predictions.append({"subject": subject_id, "predicted_vo2peak": predicted_vo2peak, "actual_vo2peak": row['vo2peak'], "scaling_factor": scaling_factor})

if not predictions:
    raise RuntimeError("No LOSO predictions were generated.")
prediction_df = pd.DataFrame(predictions)
prediction_df

In [ ]:
if "y_true" not in globals():
    y_true = prediction_df["actual_vo2peak"].to_numpy(dtype=float)
if "y_pred" not in globals():
    y_pred = prediction_df["predicted_vo2peak"].to_numpy(dtype=float)

y_true = np.asarray(y_true, dtype=float)
y_pred = np.asarray(y_pred, dtype=float)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

B = 1000
n_samples = len(y_true)
bootstrap_mae = np.empty(B)
bootstrap_rmse = np.empty(B)
bootstrap_r2 = np.empty(B)

pairs = np.column_stack([y_true, y_pred])
for i in range(B):
    sample = resample(pairs, replace=True, n_samples=n_samples)
    sample_true = sample[:, 0]
    sample_pred = sample[:, 1]
    bootstrap_mae[i] = mean_absolute_error(sample_true, sample_pred)
    bootstrap_rmse[i] = np.sqrt(mean_squared_error(sample_true, sample_pred))
    bootstrap_r2[i] = r2_score(sample_true, sample_pred)

mae_ci = np.percentile(bootstrap_mae, [2.5, 97.5])
rmse_ci = np.percentile(bootstrap_rmse, [2.5, 97.5])
r2_ci = np.percentile(bootstrap_r2, [2.5, 97.5])

print(f"MAE: {mae:.3f} (95% CI: {mae_ci[0]:.3f}, {mae_ci[1]:.3f})")
print(f"RMSE: {rmse:.3f} (95% CI: {rmse_ci[0]:.3f}, {rmse_ci[1]:.3f})")
print(f"R^2: {r2:.3f} (95% CI: {r2_ci[0]:.3f}, {r2_ci[1]:.3f})")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(prediction_df['actual_vo2peak'], prediction_df['predicted_vo2peak'], color='navy', alpha=0.75)
lims = [
    min(prediction_df['actual_vo2peak'].min(), prediction_df['predicted_vo2peak'].min()),
    max(prediction_df['actual_vo2peak'].max(), prediction_df['predicted_vo2peak'].max())
]
plt.plot(lims, lims, '--', color='gray', label='Ideal 1:1')
plt.xlabel('Actual VO2peak_abs (L/min)')
plt.ylabel('Predicted VO2peak_abs (L/min)')
plt.title('80 W Naive Baseline Predictions')
plt.xlim(lims)
plt.ylim(lims)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'naive_baseline_80w_pred_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

residuals = prediction_df['predicted_vo2peak'] - prediction_df['actual_vo2peak']
plt.figure(figsize=(6, 4))
plt.scatter(prediction_df['actual_vo2peak'], residuals, color='teal', alpha=0.75)
plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Actual VO2peak_abs (L/min)')
plt.ylabel('Residual (Predicted - Actual)')
plt.title('Residuals vs Actual')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'naive_baseline_80w_residuals_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(residuals, bins=15, color='slateblue', alpha=0.8, edgecolor='white')
plt.axvline(residuals.mean(), color='black', linestyle='--', linewidth=1, label='Mean residual')
plt.xlabel('Residual (Predicted - Actual)')
plt.ylabel('Count')
plt.title('Residual Distribution')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'naive_baseline_80w_residual_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

mean_vo2 = (prediction_df['predicted_vo2peak'] + prediction_df['actual_vo2peak']) / 2
mean_residual = residuals.mean()
std_residual = residuals.std(ddof=1)
loa = 1.96 * std_residual
plt.figure(figsize=(6, 4))
plt.scatter(mean_vo2, residuals, color='darkorange', alpha=0.75)
plt.axhline(mean_residual, color='black', linestyle='--', linewidth=1, label='Mean bias')
plt.axhline(mean_residual + loa, color='red', linestyle='--', linewidth=0.8, label='Upper LOA')
plt.axhline(mean_residual - loa, color='red', linestyle='--', linewidth=0.8, label='Lower LOA')
plt.xlabel('Mean VO2peak_abs (L/min)')
plt.ylabel('Residual (Predicted - Actual)')
plt.title('Bland-Altman Plot')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'naive_baseline_80w_bland_altman.png', dpi=300, bbox_inches='tight')
plt.show()